# 第 09 章 · 沙箱：文件、进程、网络与凭据边界

上一章获得 Permit 后，还不能直接把命令交给宿主 shell。权限需要被执行后端转换成真正可实施的文件、网络和进程边界。
“需要沙箱”“存在后端”“后端已启动且拒绝越界”是三个不同事实。

本章实验只做到 **策略描述 → 能力匹配 → 启动计划 → 交付前版本/取消检查**。
纯 Python 路径判断、mount 列表、环境变量过滤都不是 OS 沙箱；下面任何 `planned` 都带 `enforced=False`。
`Backend` 是人为设置的能力夹具，不是本机探测结果。本章不启动 Docker、bwrap、网络代理或外部服务。

真实执行入口在 [第一章 LinuxWorker](../code/mini_codex/worker.py) 的 `command()`：它构造 bwrap 命令；
第一章的 doctor/replay 才负责真实隔离及阳性对照。缺少 Linux 隔离时应记录 blocked，不能用本章通过替代它。

**运行边界**：本章使用最终包的共享组件，逐格验证本专题机制；完整任务运行见第 31 章。所有代码使用 Python 标准库；不调用真实模型、不读取凭据、不自动安装软件。按顺序运行；需要磁盘的实验只在 `fresh_run_dir()` 新建目录内写入。修改参数后，以重新执行产生的断言与输出为准。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter


In [2]:
sandbox = load_example('09', 'planner')
from dataclasses import replace

def expect_error(error_type, operation, message):
    try:
        operation()
    except error_type as error:
        assert message in str(error), str(error)
        return str(error)
    raise AssertionError(f'应出现 {error_type.__name__}: {message}')


## 1. 把抽象权限编译成可审查的计划


    读工作区、只写 out、禁止读取 secrets，需要比“workspace-write”更细的表示。
    规划器先校验规范路径，再排列规则；查询规则只解释意图，不打开文件，也不处理符号链接竞态。
    修改 `output_root`，观察写许可如何变化。deny 在本教学策略中不能被更深的 write 重新开放。


In [3]:
output_root = '/workspace/out'
spec = sandbox.Spec('policy-1', read_roots=('/workspace',), write_roots=(output_root,),
                     deny_roots=('/workspace/secrets',), network='isolated')
available = sandbox.Backend('controlled-linux-fixture', mount_rules=True, network_namespace=True)
plan = sandbox.compile_plan(spec, available)
assert plan['status'] == 'planned' and plan['enforced'] is False
assert sandbox.decision(plan, '/workspace/task.py', 'read') == 'allow'
assert sandbox.decision(plan, '/workspace/task.py', 'write') == 'deny'
assert sandbox.decision(plan, output_root + '/result.json', 'write') == 'allow'
assert sandbox.decision(plan, '/workspace/secrets/key', 'read') == 'deny'
malicious_scope = replace(spec, write_roots=(output_root, '/workspace/secrets/nested'))
assert sandbox.decision(sandbox.compile_plan(malicious_scope, available), '/workspace/secrets/nested/key', 'write') == 'deny'
plan


{'status': 'planned',
 'enforced': False,
 'backend': 'controlled-linux-fixture',
 'policy_revision': 'policy-1',
 'rules': [('/workspace', 'read'),
  ('/workspace/out', 'write'),
  ('/workspace/secrets', 'deny')],
 'network': 'isolated',
 'domains': [],
 'environment': {}}

## 2. 缺少实施能力，不能退回裸执行


    只说“沙箱模式已启用”没有意义；能力报告必须能支持所需策略。
    无文件后端或无网络 namespace 时规划直接 blocked，不能返回一个删掉限制的降级命令。
    非规范路径也先拒绝，避免同一策略对象在不同组件被解释成不同路径。


In [4]:
missing_files = replace(available, mount_rules=False)
missing_network = replace(available, network_namespace=False)
invalid_path = replace(spec, write_roots=('/workspace/../outside',))
results = [sandbox.compile_plan(spec, missing_files),
           sandbox.compile_plan(spec, missing_network),
           sandbox.compile_plan(invalid_path, available)]
assert all(r['status'] == 'blocked' for r in results)
assert all('rules' not in r for r in results)
assert [r['reason'] for r in results] == ['filesystem_backend_unavailable', 'network_boundary_unavailable', 'noncanonical_root']
results


[{'status': 'blocked', 'reason': 'filesystem_backend_unavailable'},
 {'status': 'blocked', 'reason': 'network_boundary_unavailable'},
 {'status': 'blocked', 'reason': 'noncanonical_root'}]

## 3. 代理变量不构成网络边界


    `HTTP_PROXY` 只是应用选择代理的约定。普通程序可以忽略它，直接连接 IP 或用其他协议。
    受管出口需要后端阻断直连，再由代理执行域名等策略。这里仅检查这种能力组合可否表示；
    不把一段代理 URL 视为已经建立的网络隔离。


In [5]:
allowed_domains = ('example.invalid',)  # 仅策略数据，不发请求
managed = replace(spec, network='managed', domains=allowed_domains)
proxy_only = sandbox.compile_plan(managed, available, {'HTTP_PROXY': 'http://proxy.invalid:8080'})
assert proxy_only == {'status': 'blocked', 'reason': 'managed_egress_unavailable'}
managed_backend = replace(available, managed_egress=True)
managed_plan = sandbox.compile_plan(managed, managed_backend)
assert managed_plan['status'] == 'planned' and managed_plan['enforced'] is False
assert managed_plan['domains'] == list(allowed_domains)
{'proxy_environment_only': proxy_only, 'representable_plan': managed_plan}


{'proxy_environment_only': {'status': 'blocked',
  'reason': 'managed_egress_unavailable'},
 'representable_plan': {'status': 'planned',
  'enforced': False,
  'backend': 'controlled-linux-fixture',
  'policy_revision': 'policy-1',
  'rules': [('/workspace', 'read'),
   ('/workspace/out', 'write'),
   ('/workspace/secrets', 'deny')],
  'network': 'managed',
  'domains': ['example.invalid'],
  'environment': {}}}

## 4. 工具环境不能默认继承模型凭据


    Provider 请求模型和工具访问网络是两条信任路径。为了让 subprocess 能运行而复制整个宿主环境，
    可能顺带把模型 token 交给候选脚本。最小环境应由可信执行器显式构造。
    下面仅给规划器一个人为字典，其中所有值都是公开假数据；代码没有读取 `os.environ` 或任何 `.env`。


In [6]:
supplied = {'LANG': 'C.UTF-8', 'TERM': 'dumb',
            'MODEL_API_KEY': 'PUBLIC_FAKE_MARKER',
            'HTTP_PROXY': 'http://proxy.invalid:8080',
            'PATH': '/untrusted/bin'}
environment_plan = sandbox.compile_plan(spec, available, supplied)
assert environment_plan['environment'] == {'LANG': 'C.UTF-8', 'TERM': 'dumb'}
assert 'PUBLIC_FAKE_MARKER' not in str(environment_plan)
assert supplied['MODEL_API_KEY'] == 'PUBLIC_FAKE_MARKER'  # 不修改调用方原件
{'forwarded_keys': sorted(environment_plan['environment']), 'enforced': environment_plan['enforced']}


{'forwarded_keys': ['LANG', 'TERM'], 'enforced': False}

## 5. 准备完成后仍要核对取消与政策版本


    构造计划与真正启动之间仍有时间窗口。政策改变或请求取消时，执行器不能继续使用旧准备结果。
    一次准备只能交付一次，防止两个等待者各自启动同一动作。
    `ready_for_real_executor` 只表示可交付给真实执行器，绝不是“进程已启动”或“隔离已验证”。


In [7]:
cancelled = sandbox.Preparation(plan)
cancelled.cancel()
assert cancelled.ready_for_executor('policy-1') == 'cancelled'
stale = sandbox.Preparation(plan)
assert stale.ready_for_executor('policy-2') == 'stale_policy'
assert stale.consumed is False
fresh_plan = sandbox.compile_plan(replace(spec, revision='policy-2'), available)
prepared = sandbox.Preparation(fresh_plan)
first_delivery = prepared.ready_for_executor('policy-2')
second_delivery = prepared.ready_for_executor('policy-2')
assert first_delivery == 'ready_for_real_executor'
assert second_delivery == 'already_consumed'
assert fresh_plan['enforced'] is False
{'first': first_delivery, 'duplicate': second_delivery, 'os_isolation_verified_by_this_notebook': False}


{'first': 'ready_for_real_executor',
 'duplicate': 'already_consumed',
 'os_isolation_verified_by_this_notebook': False}

## 回到 Codex 源码

以下仅陈述固定提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee` 的静态源码事实；本 Notebook 没有执行 Rust 测试。Python 类型、字段、预算与简化策略属于教学设计，不能据此声称 Codex 使用同一实现。

- [SandboxManager::select_initial; SandboxManager::transform](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/sandboxing/src/manager.rs#L346-L413)：选择需要与后端可用性不同：select_initial 可得到 None；transform 的 RequirePlatform 在无后端时返回 SandboxUnavailable。

- [SandboxManager::transform](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/sandboxing/src/manager.rs#L565-L597)：执行主机解析权限后按 backend 构造包装命令；Linux 需要 helper，managed route 加入权限封装。landlock.rs 实际序列化 profile 给默认 bwrap+seccomp helper。

- [run_main; ensure_legacy_landlock_mode_supports_policy](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/linux-sandbox/src/linux_run_main.rs#L265-L334)：Linux 默认受限文件路径先 bwrap 再内层 helper，无失败转 legacy；legacy 模式对文件受限策略明确 panic。


## 现在应能解释什么


    能区分策略可表达、后端可用、实际启动成功与越界被拒绝；能解释为什么 proxy 环境变量和 Python 路径检查不能替代 OS 边界。
    接回执行器时，计划必须由可信宿主构造，再由真实后端启动，并留存版本、拒绝与阳性对照证据。

    **未验证**：实际 bwrap/namespace、跨平台文件限制、代理直连阻断、凭据 broker、敌意进程与取消后的网络副作用。
    本章局部断言通过，只能证明规划算法，不能核销这些隔离验收项。
